In [1]:
from PreRun import PreRun, PostRun
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import make_scorer
from itertools import product
from datetime import date, datetime
from by_dates_Kfold import k_fold_split_option_a
from tqdm import tqdm
from time import sleep
from sklearn.model_selection import GridSearchCV

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [3]:
systems_cleaned.head()

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
0,2,Residential 1a,"Lakewood, CO",America/Denver,39.7214,-105.0972,1675.0,2.912,Dfb,12,...,True,True,True,False,True,multi-Si,multicrystalline_Si,PVDAQ General,2180,2011
1,3,Residential 1b,"Lakewood, CO",America/Denver,39.7214,-105.0972,1675.0,2.720,Dfb,12,...,True,True,True,False,True,amorphous si,thin_film,PVDAQ General,2094,2011
2,4,NREL x-Si -1,"Golden, CO",7,39.7406,-105.1774,1795.3,1.000,BSk,12,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,5063,2008
3,10,NREL CIS -1,"Golden, CO",7,39.7404,-105.1774,1792.8,1.120,BSk,12,...,True,True,True,True,True,cis family thin-film,thin_film,PVDAQ General,5893,2007
4,33,Silicor Materials,"Golden, CO",7,39.7404,-105.1772,1794.0,2.400,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,4438,2011


In [4]:
params = {
    'objective': 'regression',
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbose': -1
}

In [7]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))
names_list = ('inverter', 'meter', 'other')
# 1332 dropped!
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 4902, 4903]

In [6]:
start_stop_dict = {
    system_id: {
        name[0]: [0, 0] for name in name_val
    } for system_id in systems_good_timezones_manual_edit
}
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        start_stop_dict[system_id].pop(name)
        continue
    if val == 'None':
        real_val = None
    else:
        real_val = val
    prerun_system = PreRun(system_id, f'./test_results/{system_id}-{val}/', real_val, systems_cleaned)
    start_stop_dict[system_id][name][0] = prerun_system.data.at[0, 'time'].date()
    last_index = prerun_system.data.index[-1]
    start_stop_dict[system_id][name][1] = prerun_system.data.at[last_index, 'time'].date()

In [7]:
start_stop_dict

{4: {'other': [datetime.date(2007, 9, 1), datetime.date(2023, 2, 28)]},
 10: {'other': [datetime.date(2006, 1, 25), datetime.date(2023, 2, 28)]},
 33: {'other': [datetime.date(2010, 11, 10), datetime.date(2023, 2, 28)]},
 36: {'other': [datetime.date(2012, 3, 30), datetime.date(2019, 7, 21)]},
 50: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 51: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 1199: {'inverter': [datetime.date(2010, 5, 29), datetime.date(2018, 8, 3)]},
 1204: {'inverter': [datetime.date(2011, 2, 9), datetime.date(2015, 1, 5)]},
 1283: {'inverter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)],
  'meter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)]},
 1284: {'other': [datetime.date(2012, 6, 30), datetime.date(2015, 3, 15)]},
 1289: {'other': [datetime.date(2012, 9, 28), datetime.date(2020, 5, 13)]},
 4902: {'inverter': [datetime.date(2014, 7, 29), datetime.date(2018, 3, 14)],
  'meter': [datetime.date(

### Wrappers on custom objective function

In [8]:
def custom_obj(preds: np.ndarray, eval_data: lgb.Dataset):
    y_true = eval_data.get_label()
    # first derivative is (2x) * [0.5 signum(x) + 1.5]
    multiplier = 1.5 + 0.5 * np.sign(preds - y_true)
    grad = 2 * (preds - y_true) * multiplier
    hess = 2 * multiplier * np.ones_like(preds)
    return (grad, hess)


def custom_eval(preds: np.ndarray, eval_data: lgb.Dataset):
    y_true = eval_data.get_label()
    metric_name = 'weight_over'
    value = PostRun.custom_error(y_true, preds,a=1,b=2)
    is_higher_better = False
    return (metric_name, value, is_higher_better)

In [9]:
params_b = {
    'objective': custom_obj,
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbose': -1
}

In [10]:
starter_kit = PreRun(50, './test_results/50-None', None, systems_cleaned)
starter_kit.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
starter_kit.add_weather_features_only()
starter_kit.good_end_days_naive(streak=7)
good_ends = starter_kit.end_days_naive.copy(deep=True)
df = starter_kit.amended_data.copy(deep=True)
df['year'] = df['time'].dt.year
my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
df_train = df.iloc[0:int(len(df)*0.8)]
outer_cv = k_fold_split_option_a(
    df_train=df_train,
    good_ends=good_ends,
    n_splits=1,
    window_size=None,
    front_or_back='back',
    gap_day=True,
    return_type='index'
)
last_train, last_test = outer_cv[0]

In [11]:
X_tt = df_train[my_cols]
y_tt = df_train['energy']
X_test = df_train[my_cols]
y_test = df_train['energy']
lgb_last_tt = lgb.Dataset(data=X_tt,label=y_tt)
lgb_last_ho = lgb.Dataset(data=X_test,label=y_test, reference=lgb_last_tt)

In [12]:
bst_a = lgb.train(params,lgb_last_tt)
bst_b = lgb.train(params,train_set=lgb_last_tt, feval=custom_eval, valid_sets=[lgb_last_ho,])
bst_c = lgb.train(params_b, train_set=lgb_last_tt, feval=custom_eval, valid_sets=[lgb_last_ho,])

In [13]:
preds = bst_a.predict(X_test)

In [14]:
preds_b = bst_b.predict(X_test)
preds_c = bst_c.predict(X_test)

In [15]:
custom_eval(preds, lgb_last_ho)

('weight_over', np.float64(0.740840911402962), False)

In [16]:
custom_eval(preds_b, lgb_last_ho)

('weight_over', np.float64(0.740840911402962), False)

In [17]:
custom_eval(preds_c, lgb_last_ho)

('weight_over', np.float64(0.6753456742220194), False)

#### OK, so the custom objective function is doing something!

In [18]:
bst_c.dump_model(None)

{'name': 'tree',
 'version': 'v4',
 'num_class': 1,
 'num_tree_per_iteration': 1,
 'label_index': 0,
 'max_feature_idx': 9,
 'average_output': False,
 'feature_names': ['year',
  'hour_sin',
  'hour_cos',
  'day_of_year_sin',
  'day_of_year_cos',
  'last_year',
  '2_days_ago',
  'cloud_cover',
  'global_tilted_irradiance',
  'proportion_daytime'],
 'monotone_constraints': [],
 'feature_infos': {'year': {'min_value': 1994,
   'max_value': 2018,
   'values': []},
  'hour_sin': {'min_value': -1, 'max_value': 1, 'values': []},
  'hour_cos': {'min_value': -1, 'max_value': 1, 'values': []},
  'day_of_year_sin': {'min_value': -0.9999907397361901,
   'max_value': 0.9999907397361901,
   'values': []},
  'day_of_year_cos': {'min_value': -1, 'max_value': 1, 'values': []},
  'last_year': {'min_value': 0, 'max_value': 6.997999999999999, 'values': []},
  '2_days_ago': {'min_value': 0, 'max_value': 7.072975, 'values': []},
  'cloud_cover': {'min_value': 0, 'max_value': 1, 'values': []},
  'global_til

### Trial 2: LGBMRegressor variant

In [10]:
def custom_lgbm_regressor_obj(y_true: np.ndarray, y_pred: np.ndarray):
    # first derivative is (2x) * [0.5 signum(x) + 1.5]
    multiplier = 1.5 + 0.5 * np.sign(y_pred - y_true)
    grad = 2 * (y_pred - y_true) * multiplier
    hess = 2 * multiplier * np.ones_like(y_pred)
    return (grad, hess)


def custom_lgbm_regressor_eval(y_true:np.ndarray, y_pred: np.ndarray):
    metric_name = 'weight_over'
    value = PostRun.custom_error(y_true, y_pred,a=1,b=2)
    maximize = False
    return (metric_name, value, maximize)


def custom_pre_scorer(y_true, y_pred):
    return PostRun.custom_error(y_true, y_pred, a=1, b=2)

In [11]:
params_b = {
    'objective': custom_obj,
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbose': -1
}

In [21]:
trial_reg = lgb.LGBMRegressor(num_leaves=10, max_depth=10,
                              objective=custom_lgbm_regressor_obj,
                              learning_rate=0.1,
                              n_estimators=1000  # equivalent of num_iterations!
                              )
# if LightGBM 4.7.0, use eval_X and eval_y instead of eval_set!
trial_reg.fit(X=X_tt, y=y_tt, eval_metric=custom_lgbm_regressor_eval, eval_set=[(X_test, y_test)])
y_pred = trial_reg.predict(X_test)
PostRun.custom_error(y_tt, y_pred)

np.float64(0.6753456738811889)

OK, same results so far!

In [12]:
def lightgbm_one_layer(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, params: dict, streak_len: int,
                         n_splits_outer: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index')
    outer_test_results = []
    for (train_ind, test_ind) in tqdm(outer_cv):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        lgb_tt = lgb.Dataset(data=X_tt, label=y_tt)
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        lgb_ho = lgb.Dataset(X_ho, y_ho, reference=lgb_tt)
        bst = lgb.train(
            params=params,
            train_set=lgb_tt,
            feval=custom_eval,
            valid_sets=[lgb_ho,]
        )
        y_pred = bst.predict(X_ho)
        val_error = PostRun.custom_error(y_pred,y_ho,1,2)
        outer_test_results.append(val_error)
    outer_test_results = pd.Series(outer_test_results, name='custom_error')
    return outer_test_results

In [23]:
names_list = [name[0] for name in name_val]

In [24]:
good_ends_count = pd.Series(
    data = np.zeros((len(systems_good_timezones_manual_edit)*3,)),
    index=pd.MultiIndex.from_product((systems_good_timezones_manual_edit, names_list)),
    name='num_good_ends'
)
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    if val == 'None':
        real_val = None
    else:
        real_val = val
    trial_data = PreRun(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned)
    trial_data.good_end_days_naive(streak=7)
    num_good_ends = len(trial_data.end_days_naive)
    good_ends_count.at[(system_id, name)] = num_good_ends
# drop empty
good_ends_count = good_ends_count[good_ends_count > 0]
good_ends_count = good_ends_count.sort_values(ascending=False)

In [25]:
good_ends_count

50    other       2062.0
51    other       1784.0
33    other       1484.0
4     other       1127.0
10    other        891.0
1283  meter        864.0
      inverter     857.0
1199  inverter     828.0
4903  meter        452.0
1204  inverter     428.0
4903  inverter     427.0
4902  meter        394.0
      inverter     374.0
36    other        358.0
1289  other        322.0
1284  other        177.0
Name: num_good_ends, dtype: float64

In [26]:
early_results = lightgbm_one_layer(
    50, './test_results/50-None', None, systems_cleaned, params_b, 7, 100
)

100%|██████████| 100/100 [07:49<00:00,  4.69s/it]


In [27]:
early_results.describe()

count    100.000000
mean       1.273282
std        1.165655
min        0.029381
25%        0.392297
50%        0.888917
75%        1.778331
max        5.927781
Name: custom_error, dtype: float64

### Practice sklearn.API variants of commands

In [26]:
params_b = {
    'objective': custom_obj,
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metric': 'custom',
    'verbose': -1
}

In [27]:
def lightgbm_one_layer_b(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, params: dict, streak_len: int,
                         n_splits_outer: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index')
    outer_test_results = []
    for (train_ind, test_ind) in tqdm(outer_cv):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        my_regressor = lgb.LGBMRegressor(num_leaves=params['num_leaves'], 
                                         learning_rate=params['learning_rate'],
                                         objective=custom_lgbm_regressor_obj,
                                         max_depth=params['max_depth'],
                                         n_estimators=params['num_iterations'])
        my_regressor.fit(X_tt, y_tt, eval_metric=custom_lgbm_regressor_eval, eval_set=(X_ho, y_ho))
        lgb_tt = lgb.Dataset(data=X_tt, label=y_tt)
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        lgb_ho = lgb.Dataset(X_ho, y_ho, reference=lgb_tt)
        bst = lgb.train(
            params=params,
            train_set=lgb_tt,
            feval=custom_eval,
            valid_sets=[lgb_ho,]
        )
        y_pred = bst.predict(X_ho)
        val_error = PostRun.custom_error(y_pred,y_ho,1,2)
        outer_test_results.append(val_error)
    outer_test_results = pd.Series(outer_test_results, name='custom_error')
    return outer_test_results

In [30]:
early_results_b = lightgbm_one_layer_b(
    50, './test_results/50-None', None, systems_cleaned, params_b, 7, 100
)

100%|██████████| 100/100 [13:29<00:00,  8.09s/it]


In [31]:
mse(early_results,early_results_b)

8.319729439552853e-24

In [32]:
np.mean(np.abs(early_results - early_results_b))

np.float64(9.235544273789032e-13)

#### Not perfect, but decidedly close enough!

In [33]:
early_results = np.asarray(early_results)

In [34]:
systems_cleaned[systems_cleaned['system_id']==50]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
8,50,NREL x-Si 6,"Golden, CO",7,39.742,-105.1727,1994.7,6.0,BSk,12,...,True,True,True,True,True,Unknown,unknown,PVDAQ General,8455,1995


In [35]:
terms = [early_results.mean(), early_results.std(), early_results.min(), early_results.max()]

In [36]:
print(f'Mean: {terms[0]}, std: {terms[1]}, range:{terms[2]}, {terms[3]}')

Mean: 1.2732820683285153, std: 1.1598119460555265, range:0.029380985558968016, 5.927781187257565


## Add in some hyperparameter tuning!

In [28]:
def better_lightbgm_test(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame,
                         streak_len: int,
                         n_splits_outer: int, n_splits_inner: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train_max = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train_max,
        good_ends=good_ends,
        n_splits=n_splits_outer, 
        window_size=None, 
        front_or_back='back',
        gap_day=False,
        return_type='index')
    outer_results = np.zeros((len(outer_cv),))
    for i, (train_ind, test_ind) in enumerate(outer_cv):
        df_train_out = df_train.loc[train_ind]
        df_val_out = df_train.loc[test_ind]
        X_train_out = df_train_out[my_cols]
        y_train_out = df_train_out['energy']
        lgb_train_out = lgb.Dataset(data=X_train_out, label=y_train_out)
        X_val_out = df_val_out[my_cols]
        y_val_out = df_val_out['energy']
        lgb_val_out = lgb.Dataset(X_val_out, y_val_out, reference=lgb_train_out)
        # inner loop,
        inner_cv = k_fold_split_option_a(
            df_train=df_train_out,
            good_ends=good_ends,
            n_splits=n_splits_inner,
            window_size=None, 
            front_or_back='back',
            gap_day=False,
            return_type='index')
        param_grid = {
            'num_leaves': (7, 15, 31),
            'max_depth': (-1, 5, 7, 10),
            'learning_rate': (0.05, 0.1, 0.15, 0.2),
            'n_estimators': (10, 100, 1000),
            'subsample': (0.8, 1.0),
            'colsample_bytree': (0.8, 1.0),
        }
        in_regressor = lgb.LGBMRegressor()
        local_scorer = make_scorer(custom_pre_scorer, greater_is_better=False)
        searcher = GridSearchCV(in_regressor, param_grid, scoring=local_scorer, cv=inner_cv,)
        searcher.fit(X_train_out, y_train_out)
        print(i)
        print(searcher.cv_results_)
        bst = searcher.best_estimator_
        y_out_preds = bst.predict(X_val_out)
        outer_results[i] = PostRun.custom_error(y_test, y_out_preds)
    outer_results = pd.Series(outer_results, name = ['outer_error'])
    return outer_results

This code takes way too long to run.  Will need to adjust

In [66]:
early_results_c = better_lightbgm_test(
    50, './test_results/50-None', None, systems_cleaned, 7, 15, 15
)

KeyboardInterrupt: 

In [13]:
def lightgbm_one_layer_c(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, streak_len: int,
                         n_splits_outer: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index')
    outer_test_results = []
    for i, (train_ind, test_ind) in enumerate(outer_cv):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        param_grid = {
            'num_leaves': (7, 15, 31),
            'max_depth': (-1, 5, 7, 10),
            'learning_rate': (0.05, 0.1, 0.15, 0.2),
            'n_estimators': (10, 100, 1000),
            'subsample': (0.8, 1.0),
            'colsample_bytree': (0.8, 1.0),
        }
        out_regressor = lgb.LGBMRegressor()
        local_scorer = make_scorer(custom_pre_scorer, greater_is_better=False)
        searcher = GridSearchCV(out_regressor, param_grid, scoring=local_scorer, cv=5,)
        searcher.fit(X_tt, y_tt)
        print(i)
        print(searcher.cv_results_)
        bst = searcher.best_estimator_
        y_out_preds = bst.predict(X_ho)
        outer_results[i] = PostRun.custom_error(y_ho, y_out_preds)
    outer_test_results = pd.Series(outer_test_results, name='custom_error')
    return outer_test_results

In [34]:
from pathlib import Path

In [36]:
my_dir = Path('../../../../data_ds_project/parquet_cleaned_energy/')
print(my_dir.is_dir())

True


In [ ]:
early_results_c = lightgbm_one_layer_c(
    50, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 7, 15
)

0
{'mean_fit_time': array([0.03102789, 0.03211222, 0.03967242, 0.04147296, 0.04286056,
       0.04258604, 0.09079652, 0.09231019, 0.12193203, 0.12164493,
       0.17376943, 0.17092075, 0.57686481, 0.56076007, 0.7986402 ,
       0.79924455, 1.26065321, 1.28357515, 0.03828282, 0.03780603,
       0.04218712, 0.04166288, 0.04169049, 0.04099021, 0.08867059,
       0.08740745, 0.11910162, 0.11834989, 0.15390935, 0.34904985,
       1.19694662, 1.59443202, 2.84829588, 2.35025587, 1.02361298,
       0.98830366, 0.0350256 , 0.0328207 , 0.03908868, 0.03742471,
       0.04467998, 0.04508343, 0.08413692, 0.09433937, 0.1268424 ,
       0.12845035, 0.19571247, 0.18470583, 0.53468256, 0.56527352,
       0.88137488, 1.08014803, 2.67561135, 5.22110395, 0.03402662,
       0.0306675 , 0.03765669, 0.0395772 , 0.04661589, 0.04645443,
       0.08151588, 0.08260794, 0.12801814, 0.12774816, 0.20246186,
       0.21224775, 0.51792603, 0.5323463 , 0.83506889, 2.10344963,
       3.66268401, 5.8254735 , 0.06053624,

ValueError: operands could not be broadcast together with shapes (58534,) (14,) 

In [20]:
def lightgbm_one_layer_d(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, streak_len: int,
                         n_splits_outer: int, sample_spacing: int):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=2, include_last_year=True, include_hour_cyclic=True, include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    prerun_system.good_end_days_naive(streak=streak_len)
    good_ends = prerun_system.end_days_naive.copy(deep=True)
    df = prerun_system.amended_data.copy(deep=True)
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '2_days_ago', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_train = df.iloc[0:int(len(df)*0.8)]
    outer_cv = k_fold_split_option_a(
        df_train=df_train,
        good_ends=good_ends,
        n_splits=n_splits_outer,
        window_size=None,
        front_or_back='back',
        gap_day=True,
        return_type='index',
        sample_spacing=sample_spacing)
    param_grid = {
        'num_leaves': (7, 15, 31),
        'max_depth': (-1, 5, 7),
        'learning_rate': (0.1, 0.2),
        'n_estimators': (100,),  # wanted more, but there is literally no time!
        'subsample': (0.8, 1.0),
        'colsample_bytree': (0.8, 1.0),
    }
    col_names = [str(param_settings) for param_settings in product(
        param_grid['num_leaves'],
        param_grid['max_depth'],
        param_grid['learning_rate'],
        param_grid['n_estimators'],
        param_grid['subsample'],
        param_grid['colsample_bytree'])]
    outer_test_results = pd.DataFrame(
        np.zeros((len(outer_cv), len(col_names))),
        columns=col_names
    )
    for i, (train_ind, test_ind) in enumerate(tqdm(outer_cv)):
        df_tt = df_train.loc[train_ind]
        df_ho = df_train.loc[test_ind]
        X_tt = df_tt[my_cols]
        y_tt = df_tt['energy']
        lgb_tt = lgb.Dataset(data=X_tt, label=y_tt)
        X_ho = df_ho[my_cols]
        y_ho = df_ho['energy']
        lgb_ho = lgb.Dataset(data=X_ho, label=y_ho, reference=lgb_tt)
        for j, param_settings in enumerate(product(param_grid['num_leaves'],
            param_grid['max_depth'],
            param_grid['learning_rate'],
            param_grid['n_estimators'],
            param_grid['subsample'],
            param_grid['colsample_bytree']
        )):
            temp_params = {
                'objective': custom_obj,
                'task': 'train',
                'num_leaves': param_settings[0],
                'max_depth': param_settings[1],
                'learning_rate': param_settings[2],
                'num_iterations': param_settings[3],
                'subsample': param_settings[4],
                'colsample_bytree': param_settings[5],
                'metric': 'custom',
                'verbose': -1
            }
            bst = lgb.train(
                params=temp_params,
                train_set=lgb_tt,
                feval=custom_eval,
                valid_sets=[lgb_ho,]
            )
            y_pred = bst.predict(X_ho)
            val_error = PostRun.custom_error(y_pred,y_ho,1,2)
            outer_test_results.at[i, col_names[j]] = val_error
    outer_test_results.to_csv(f'./testing_many_folds/{system_id}_{met_or_inv}.csv', index=False)
    return outer_test_results

In [ ]:
# full results for System 50 to see which one(s) are best.
early_results_d = lightgbm_one_layer_d(
    50, '../../../../data_ds_project/parquet_cleaned_energy/', None, systems_cleaned, 7, -1, 5
)

 42%|████▏     | 123/294 [28:11<45:18, 15.90s/it] 